# Rebuilding SMART-II Test Set (Patient-wise Split)

This notebook splits the AI Khosh SMART-II dataset by ensuring that images from the same patient do not leak across the `train`, `val`, and `test` sets.

It uses `Patient's Metadata - SMART - SMITA.xlsx` to map each image to its Patient ID.

**Source Directory**: `/Users/alokdwivedi/dev/oral-images-dataset/SMART-II-Dataset`
**Output Directory**: `/Users/alokdwivedi/dev/oral_cavity_smart_ii_patientwise`

In [8]:
import os
import shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
import json
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)

SOURCE_DIR = "/Users/alokdwivedi/dev/oral-images-dataset/SMART-II-Dataset"
OUTPUT_DIR = "/Users/alokdwivedi/dev/oral_cavity_smart_ii_patientwise"

print(f"Source: {SOURCE_DIR}")
print(f"Target: {OUTPUT_DIR}")

Source: /Users/alokdwivedi/dev/oral-images-dataset/SMART-II-Dataset
Target: /Users/alokdwivedi/dev/oral_cavity_smart_ii_patientwise


## 1. Load Data

In [9]:
metadata_path = os.path.join(SOURCE_DIR, "Patient's Metadata - SMART - SMITA.xlsx")
metadata_df = pd.read_excel(metadata_path)
metadata_df.columns = metadata_df.columns.str.strip()
display(metadata_df.head())
print(f"Total metadata rows: {len(metadata_df)}")

,S.No,SMITA ID,Age in years,Sex,Religion,Marital status,Education,Occupation,Diet history,Spice consumption,...,Margin,Surface feature,Description of the Lesion,"If multiple lesion present, mention number of sites involved",Size in cms (Length x Width),Associated with pain,"If yes, duration of pain (in days)",Type of pain,Lesion classification,Others
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,SMITA00363,54.0,Male,Hindu,Married,Primary,Unskilled worker,Non-Vegetarian,Moderate,...,-,-,-,-,-,-,-,-,-,-
2,2.0,SMITA00364,18.0,Male,Hindu,Unmarried,Secondary,Not working,Non-Vegetarian,Moderate,...,-,-,-,-,-,-,-,-,-,-
3,3.0,SMITA00365,39.0,Female,Hindu,Married,Primary,Unskilled worker,Non-Vegetarian,Moderate,...,-,-,-,-,-,-,-,-,-,-
4,4.0,SMITA00366,42.0,Male,Hindu,Married,Primary,Unskilled worker,Non-Vegetarian,Moderate,...,Not well demarcated,Smooth,single,-,2x1,no,-,-,No OPMD,Linea alba


Total metadata rows: 136


## 2. Scan Directory for Images

In [10]:
image_records = []
for category_folder in ['01. Normal', '02. Variation', '03. OPMD']:
    folder_path = os.path.join(SOURCE_DIR, category_folder)
    if not os.path.exists(folder_path): continue
    # We must walk recursively because patient images are nested in subfolders
    for root, dirs, files in os.walk(folder_path):
        for file_name in files:
            if file_name.startswith('.'): continue # Skip hidden files like .DS_Store
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                image_records.append({
                    'Original_Folder': category_folder,
                    'File_Name': file_name,
                    'Full_Path': os.path.join(root, file_name)
                })
image_df = pd.DataFrame(image_records)
print(f"Total image files found: {len(image_df)}")

Total image files found: 2148


## 3. Merge Metadata with Images

In [11]:
def extract_patient_id(filename):
    base = os.path.splitext(filename)[0]
    parts = base.split('_')
    if len(parts) > 1: return parts[0]
    parts = base.split('-')
    if len(parts) > 1: return parts[0]
    return base

image_df['Patient ID'] = image_df['File_Name'].apply(extract_patient_id)
print(f"Extracted {image_df['Patient ID'].nunique()} unique patients from filenames.")

class_mapping = {
    '01. Normal': 'Healthy',
    '02. Variation': 'Benign',
    '03. OPMD': 'OPMD'
}
image_df['target_class'] = image_df['Original_Folder'].map(class_mapping)
display(image_df.head())

Extracted 135 unique patients from filenames.


,Original_Folder,File_Name,Full_Path,Patient ID,target_class
0,01. Normal,SMITA00368_R_LA.jpg,/Users/alokdwivedi/dev/oral-images-dataset/SMA...,SMITA00368,Healthy
1,01. Normal,SMITA00368_R_UA.jpg,/Users/alokdwivedi/dev/oral-images-dataset/SMA...,SMITA00368,Healthy
2,01. Normal,SMITA00368_R_RB.jpg,/Users/alokdwivedi/dev/oral-images-dataset/SMA...,SMITA00368,Healthy
3,01. Normal,SMITA00368_R_DT.jpg,/Users/alokdwivedi/dev/oral-images-dataset/SMA...,SMITA00368,Healthy
4,01. Normal,SMITA00368_R_UL.jpg,/Users/alokdwivedi/dev/oral-images-dataset/SMA...,SMITA00368,Healthy


## 4. Patient-wise Split

In [12]:
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
train_idx, temp_idx = next(gss_train.split(image_df, groups=image_df['Patient ID']))
train_df = image_df.iloc[train_idx].copy()
temp_df = image_df.iloc[temp_idx].copy()

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['Patient ID']))
val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_df['split'], val_df['split'], test_df['split'] = 'train', 'val', 'test'
final_df = pd.concat([train_df, val_df, test_df])

print("Split sizes (images):")
print(final_df['split'].value_counts())
print("\nSplit sizes (patients):")
print(final_df.groupby('split')['Patient ID'].nunique())

Split sizes (images):
split
train    1496
test      332
val       320
Name: count, dtype: int64

Split sizes (patients):
split
test     21
train    94
val      20
Name: Patient ID, dtype: int64


## 5. Copy Files

In [13]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
for split in ['train', 'val', 'test']:
    for cat in set(class_mapping.values()):
        os.makedirs(os.path.join(OUTPUT_DIR, split, cat), exist_ok=True)

print("Copying files...")
for _, row in tqdm(final_df.iterrows(), total=len(final_df)):
    shutil.copy2(row['Full_Path'], os.path.join(OUTPUT_DIR, row['split'], row['target_class'], row['File_Name']))
print(f"Dataset successfully created at {OUTPUT_DIR}")

Copying files...


  0%|          | 0/2148 [00:00<?, ?it/s]

Dataset successfully created at /Users/alokdwivedi/dev/oral_cavity_smart_ii_patientwise


## 6. Save Manifests

In [14]:
manifest_list = [os.path.join(OUTPUT_DIR, 'test', row['target_class'], row['File_Name']) for _, row in test_df.iterrows()]
manifest_path = os.path.join(OUTPUT_DIR, 'SMART_II_patientwise_test_manifest.json')
with open(manifest_path, 'w') as f: json.dump(manifest_list, f, indent=2)
print(f"Saved test manifest to {manifest_path} with {len(manifest_list)} images.")

Saved test manifest to /Users/alokdwivedi/dev/oral_cavity_smart_ii_patientwise/SMART_II_patientwise_test_manifest.json with 332 images.
